# ETCC解析の流れ：2026-11-24 run00〜07

外部JSONを入口に、保存済み結果の確認と再計算を行う。上から順に実行できる。初期設定では**保存済み結果の表示のみ**で、SSDのraw走査を開始しない。作成中の結果は「未作成」と表示する。

イベント選別は **ビーム軸との3D距離 ≤ 10 mm → Z側のイベント除外 → raw HAヒット数が1または2** の順。1ヒットと2ヒットは最後に分岐する。ピクセル有効性はこの後にHG/LGで独立に判定する。

Z側条件は「頂点Z > 15 mm」だけではなく、**どちらかのTPCボードにZ ≤ 15 mmのrawヒットが一つでもあればイベント全体を除外**する。追加のXY・高Z側fiducial条件は入れない。


In [ ]:
from pathlib import Path  # リポジトリと結果のパスを扱う。
import json  # 外部設定と件数表を読む。
import subprocess  # 既存スクリプトを同じPythonで実行する。
from html import escape  # パスや表の文字列を安全に表示する。
from IPython.display import display, Image, HTML, Markdown, FileLink  # 画像とリンクをNotebookに表示する。
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ETCC_cal/settings.py').is_file()), None)  # 起動位置からリポジトリを探す。
if REPO is None:  # リポジトリ外での起動を検出する。
    raise RuntimeError('QLリポジトリまたはETCC_cal/notebooksを作業ディレクトリにしてください。')  # 必要な起動場所を示す。
CONFIG = REPO / 'ETCC_cal/config/20261124_run00_07_stages.json'  # 入出力・ラン・カットの設定はこのJSONを編集する。
PYTHON = '/usr/bin/python3'  # 既存解析で検証したsystem Pythonを使う。
REBUILD_FROM_CACHE = False  # Trueなら保存キャッシュからシンチ図を再計算し、設定先の結果を更新する。
REBUILD_RAW = False  # TrueならSSDを読む全工程を実行する。長時間かかるため通常はFalseのままにする。
config = json.loads(CONFIG.read_text())  # 外部設定を読む。
def path(value):  # 相対パスをリポジトリ基準で解決する。
    return REPO / value  # 絶対パスならPathがそのまま保持する。
RESULT = path(config['result_dir'])  # 保存先をJSONから決める。
def read_json(file):  # 未作成の集計も表示処理を止めずに扱う。
    return json.loads(file.read_text()) if file.is_file() else None  # ある場合だけ読む。
def show_image(file, width=1200):  # 未作成と保存済み画像を区別して表示する。
    display(Image(filename=str(file), width=width)) if file.is_file() else display(Markdown(f'未作成：`{file}`'))  # 画像をNotebookに埋め込む。
def run_script(script, *options):  # 解析の実装は既存Pythonに集約する。
    command = [PYTHON, '-u', str(REPO / 'ETCC_cal' / script), '--config', str(CONFIG), *options]  # shell展開なしで引数を渡す。
    print('実行:', ' '.join(command), flush=True)  # 実行内容を確認できるようにする。
    subprocess.run(command, cwd=REPO, check=True)  # エラーを隠さず、そのセルで停止する。
display(Markdown(f'設定：`{CONFIG}`\n\n保存先：`{RESULT}`'))  # 実際の入口を表示する。
print('runs:', ', '.join(config['runs']))  # 対象ランを確認する。
print('drift velocity [mm/clock]:', config['velocity'], '; clock offset [clock]:', config['clock_offset'])  # 座標変換を確認する。
print('beam radius [mm]:', config['polarization']['beam_max_distance_mm'], '; Z veto [mm]:', config['z_veto'])  # 選別の単位を表示する。


## 1. 入力と再計算

`REBUILD_RAW=True` は、全頂点・Z除外・HG/LG重心の準備、各段階のシンチ図、1/2ヒット解析、TPC段階別積算を順に実行する。完了キャッシュは再利用する。既に解析を実行中なら重複起動せず、Falseのまま後続セルで結果を確認する。

run00〜05の頂点は早い16クロックレコードを使う従来定義を維持し、10万イベントずつ結合する。TPC積算では全有効クロック0〜1023を使う。段階別解析は**完成rawのみ**。別途作成済みの「全データ・カットなし積算」は末尾tmpを含むため、両者の総ヒット数は一致するとは限らない。

ビーム距離は既存のrun06・07の局所ピークfit直線を共通に使用する。図中の星印は底面シンチZ=346.06 mm平面での直線通過点。軸の重ね描き自体はイベント選別ではない。


In [ ]:
required = {'全頂点CSV': path(config['inputs']['vertices_all']), '保存ビーム軸': path(config['inputs']['polarization_beam_axis']), 'シンチ幾何': path(config['inputs']['scintillator_geometry']), 'HG位置較正': path(config['inputs']['ha_position_calibration_dir'])}  # 保存済み入力だけを確認する。
for name, file in required.items():  # SSDの広域探索は行わない。
    print(('OK' if file.exists() else '未作成/参照不能'), name, file)  # 必要な入力の状態を表示する。
if REBUILD_RAW:  # 利用者がraw再計算を選んだ場合だけ実行する。
    run_script('run_stage_workflow.py')  # 準備から全段階まで実行し、詳細はshared/workflow_logsへ保存する。
else:  # 通常は保存済み結果を表示する。
    print('raw再計算は実行しません。')  # 重い処理が起動していないことを明示する。
if REBUILD_FROM_CACHE and not REBUILD_RAW:  # rawを読まずにシンチ関連だけ更新する。
    run_script('stage_pixel_maps.py')  # 各累積条件のHG/LGピクセル図を保存する。
    run_script('pixel_polarization.py', '--from-cache')  # raw2ヒットの方位角とピクセル図を保存する。
    run_script('single_hit_pixels.py')  # raw1ヒットのHG/LG図を保存する。
    run_script('stage_flag_quality_maps.py')  # stage03を全rawフラグ、stage04をgain品質選別後として保存する。


## 2. 各カットで残る件数

シンチ表の分母は**raw HAヒットが1個以上あるイベント**。TPC表の分母は**全ボードの記録TPCイベントの和集合**で、ヒットなしも含む。分母を混同しない。シンチ図の表示点数はHG/LGの有効ピクセル数で、イベント数とは異なる。2ヒット図は両ピクセルが有効で、方位角を定義できるペアを使う。


In [ ]:
stages = [('00_before_cuts', 'カット前'), ('01_beam_axis', 'ビーム軸距離'), ('02_after_z0', 'さらにZ側除外'), ('03_raw1', 'さらにraw 1ヒット'), ('03_raw2', 'さらにraw 2ヒット')]  # 最後の2段階は並列の分岐を表す。
def show_cutflow(file, title):  # 保存された件数を表にまとめる。
    saved = read_json(file)  # 集計を読み込む。
    if saved is None:  # まだ解析が終わっていない場合を扱う。
        display(Markdown(f'{title}：未作成 (`{file}`)'))  # 作成中の結果を捏造しない。
        return  # 次の表示へ進む。
    rows = saved['runs']  # ランごとの件数を取り出す。
    body = ''.join('<tr><td>'+escape(row['run'])+'</td>'+''.join(f"<td>{row['events'][key]:,}</td>" for key, _ in stages)+'</tr>' for row in rows)  # イベント数だけを表示する。
    body += '<tr><td>合計</td>'+''.join(f"<td>{sum(row['events'][key] for row in rows):,}</td>" for key, _ in stages)+'</tr>'  # ラン間は件数だけを合算する。
    display(HTML('<h3>'+escape(title)+'</h3><table><tr><th>run</th>'+''.join('<th>'+label+'</th>' for _, label in stages)+'</tr>'+body+'</table>'))  # 件数表を表示する。
show_cutflow(RESULT / 'stage_cutflow.json', 'シンチ：raw HAヒットありを分母にした件数')  # ピクセル有効性を要求する前の件数を表示する。
show_cutflow(RESULT / 'stage_track_cutflow.json', 'TPC：全記録イベントを分母にした件数')  # TPCの分母を独立に表示する。


## 3. stage00〜03：品質選別前のrawフラグ位置図

ビーム軸→Z側→raw1/2の順に表示する。各段階で全rawフラグを残し、HG/LGの重心から位置を求められるものはピクセル位置、未決定のものはch中心に置く。飽和でヒットを落とすのは次のstage04。HG較正のLGへの流用は暫定で、飽和した重心の推定位置にも歪みがあり得る。


In [ ]:
groups = ['without_z0_cut/stages/00_before_cuts', 'without_z0_cut/stages/01_beam_axis', 'with_z0_cut/stages/02_beam_axis_then_z0', 'with_z0_cut/stages/03_raw_single_hit', 'with_z0_cut/stages/03_raw_two_hits']  # 累積条件に対応した保存先を指定する。
for (_, label), folder in zip(stages, groups):  # カットの順番で表示する。
    display(Markdown(f'### {label}'))  # どの段階か示す。
    for gain in ('hg', 'lg'):  # HG/LGを両方表示する。
        show_image(RESULT / folder / f'{gain}.png')  # 各段階のXY・XZ・YZ図を表示する。


## stage04：HG/LGの品質選別後
直前のstage03は全rawフラグを残し、位置未決定ならch中心に置く。ここで初めてHG/LG別の飽和・正積分・有限な重心・較正範囲・有効ペアなどを選別する。


In [ ]:
quality_flow = read_json(RESULT / 'stage_flag_quality_cutflow.json')  # rawフラグと品質選別の件数を読む。
if quality_flow:  # 両stageが作成済みなら件数の違いも示す。
    for branch, record in quality_flow['branches'].items():  # raw1/2を別々に確認する。
        print(f"raw {branch}: stage03={record['raw_events']:,} events")  # 品質選別前の共通分母を表示する。
        for gain, counts in record['gains'].items():  # HG/LGそれぞれの減少数を確認する。
            print(gain.upper(), 'stage04 events:', counts['after_quality_events'], '; stage03 channel-centre hits:', counts['raw_channel_centre_hits'])  # 採用数と代替位置の数を混同しない。
for branch in ('single_hit', 'two_hits'):  # raw1/2の分岐ごとに品質選別後を表示する。
    display(Markdown(f'### stage04：{branch}'))  # stage03とは独立した段階名を示す。
    for gain in ('hg', 'lg'):  # gain別の飽和などを適用した図を表示する。
        show_image(RESULT / f'with_z0_cut/stages/04_{branch}_quality/{gain}.png')  # ピクセル位置を表示する。


## 4. 段階別のTPC飛跡積算

各通過イベントの**全rawストリップ・クロックヒット**を足す。頂点1票や3D点の全組み合わせではない。raw1/2段階でもHG/LGのピクセル有効性は要求しないので、シンチ図とTPC図の採用数には差がある。

カラー調整版は空ビンを除く10〜99.5百分位、全レンジ版は下限1のLogNorm。色範囲は各図で変わるため、色だけで絶対量を比較しない。絶対数は件数表・JSONで確認する。

stage04のTPC積算は追加処理後に各品質stageのtpc_hg/とtpc_lg/へ保存する。シンチ図と同じ採用イベントでXZ/YZ/コインシデンスXYを表示する。

3DコインシデンスXY追加処理が完了すると同じ画像がXZ・YZ・XYの3パネルになる。XYは同イベント・同uPIC・同クロックのX×Y全組合せ（クロック差0）で、最初の頂点だけではない。XZ/YZのraw stripヒット数とXYの組合せ数は色バーの単位を分ける。


In [ ]:
for (_, label), folder in zip(stages, groups):  # TPC積算を同じカット順に表示する。
    display(Markdown(f'### {label}：TPC'))  # シンチ図との違いを明示する。
    show_image(RESULT / folder / 'tpc_tracks_z_contrast.png')  # 全Z範囲のカラー調整版を表示する。
    full = RESULT / folder / 'tpc_tracks_z_fullscale.png'  # 全レンジ版も参照できるようにする。
    if full.is_file():  # 作成済みの場合だけリンクを表示する。
        display(FileLink(str(full)))  # カラーレンジを比較する入口にする。


## 5. raw2ヒットの方位角分布

アクセプタンス補正前の分布であり、fitの変調度をそのまま偏光度と解釈しない。HG/LGは標準では独立に有効ペアを採用する。条件は外部JSONの`polarization`で指定する。


In [ ]:
prefix = path(config['outputs']['pixel_polarization_prefix'])  # 方位角解析の出力名をJSONから読む。
for gain in ('hg', 'lg'):  # HG/LGの2種類を確認する。
    show_image(Path(str(prefix) + f'_{gain}.png'))  # 方位角・ピクセル位置・raw多重度を表示する。


## 6. 保存済み飛跡例とビームfit

以下はrun06の保存済み候補。Z側除外を通過したことは確認済みだが、今回の全カットを通過した新しい代表例の自動抽出ではない。新しいrun00〜05の例は別途候補を選ぶ必要がある。HTMLは回転できる3D表示。


In [ ]:
REFERENCE = REPO / 'ETCC_cal/output/20261124_run06_07'  # 保存済みの候補と軸の場所を指定する。
show_image(REFERENCE / 'with_z0_cut/beam_axis/axis_z0veto.png')  # PCAでなく局所ピークのfit結果を表示する。
for trigger in (22204, 55668):  # 既に確認した2例を表示する。
    stem = REFERENCE / f'with_z0_cut/events/run06_trig{trigger}'  # 候補の保存名を作る。
    show_image(stem.with_suffix('.png'))  # TPCとシンチ点を重ねた保存画像を表示する。
    if stem.with_suffix('.html').is_file():  # 3D表示が保存されているか確認する。
        display(FileLink(str(stem.with_suffix('.html'))))  # 回転可能なHTMLへの入口を表示する。


## 7. 全データ・カットなし積算と座標

既存の全データ積算は末尾tmpの読める部分も含む。上の段階別完成raw解析とは別の結果として確認する。

Z [mm] = 0.39358 [mm/clock] × (clock − 133.69 [clock]) + 5 [mm]。GEM基準より早いクロックもrawには入るためZが負になる。負Zは計算ミスだけを意味するものではなく、時刻オフセットや信号の由来を確認する対象。

散乱点からのドリフト速度推定は今後の検討。カットなしの頂点clock分布で物理端と選別・読み出しの影響を確認してからfitする。このNotebookでは新しい速度を算出しない。


In [ ]:
show_image(REPO / 'ETCC_cal/output/20261124_run00_07/without_z0_cut/tracks/all_runs_z_contrast.png')  # tmp込みの既存の全データ積算を表示する。
log_dir = RESULT / 'shared/workflow_logs'  # 全工程のログ保存先を指定する。
for file in sorted(log_dir.glob('*.log')):  # 全段階の処理状況を確認する。
    print('\n', file.name, '\n', '\n'.join(file.read_text(errors='replace').splitlines()[-3:]))  # 最新3行だけ表示する。


## stage04：品質選別後のTPC積算
シンチ採用CSVと同じイベントで積算する。生成中は未作成と表示する。


In [ ]:
for branch in ('single_hit','two_hits'):  # raw1/2の品質選別後を分ける。
    for gain in ('hg','lg'):  # HG/LGごとの採用イベントを使ったTPC図を表示する。
        display(Markdown(f'### stage04 {branch} {gain.upper()}：TPC'))  # 条件を見出しにする。
        show_image(RESULT / f'with_z0_cut/stages/04_{branch}_quality/tpc_{gain}/tpc_tracks_z_contrast.png')  # XZ・YZ・3D同時ヒットXYを確認する。


## stage04：TPC積算の水平／円偏光比
各投影の総数で規格化した形状比。完成後に保存図を表示する（raw再走査なし）。


In [ ]:
for branch in ('single_hit','two_hits'):  # raw1/2の品質選別後を分ける。
    for gain in ('hg','lg'):  # gainごとの水平／円TPC比を確認する。
        display(Markdown(f'### TPC水平／円：{branch} {gain.upper()}'))  # 条件を見出しにする。
        show_image(REPO / f'ETCC_cal/output/20261124_horizontal_vs_circular/stage04_tpc_ratio/04_{branch}_quality/{gain}_tpc_ratio_z.png')  # 未完成なら未作成と表示する。


## stage04シンチ方位角の水平／円偏光比
上段は規格化分布、下段は比。左HG、右LG。誤差棒は統計誤差。


In [ ]:
show_image(REPO / 'ETCC_cal/output/20261124_horizontal_vs_circular/stage04_double_hit_quality_azimuth_ratio/azimuth_ratio.png')  # 保存済み方位角比を表示する。
